# Helm 04: testing a chart and handling secrets

Without a cluster you can lint, render with every values file, validate the output against Kubernetes schemas and diff environments. Secrets enter through references resolved at render time or through an operator, never as values in git.


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && [ -d gitops-renderers ] || git clone -q --recurse-submodules https://github.com/cznewt/gitops-renderers.git
cd /source/work/gitops-renderers/examples/02-helm && export HOME=/tmp && helm dependency update web >/dev/null 2>&1; helm lint --strict web -f values-prod.yaml


In [ ]:
cd /source/work/gitops-renderers/examples/02-helm
for env in dev prod; do helm template web web -n web --skip-tests -f values-$env.yaml > /tmp/helm-$env.yaml; done; kubeconform -strict -ignore-missing-schemas -summary /tmp/helm-dev.yaml /tmp/helm-prod.yaml


In [ ]:
diff <(yq 'select(.kind == "Deployment" and .metadata.name == "web") | .spec' /tmp/helm-dev.yaml) <(yq 'select(.kind == "Deployment" and .metadata.name == "web") | .spec' /tmp/helm-prod.yaml) || true


`vals` resolves `ref+sops://`, `ref+vault://`, `ref+awssecrets://` and friends inside a values file; helm-secrets is a plugin around the same engine. The `ExternalSecret` the chart emits is the pattern for real clusters.


In [ ]:
cd /source/work/gitops-renderers
export HOME=/tmp SOPS_AGE_KEY_FILE=$PWD/examples/secrets/demo.agekey
cat examples/secrets/values-prod-secrets.yaml; echo ---; vals eval -f examples/secrets/values-prod-secrets.yaml


In [ ]:
cd /source/work/gitops-renderers
export HOME=/tmp SOPS_AGE_KEY_FILE=$PWD/examples/secrets/demo.agekey
vals eval -f examples/secrets/values-prod-secrets.yaml > /tmp/secrets.yaml && helm template web examples/02-helm/web -n web --skip-tests -f examples/02-helm/values-prod.yaml -f /tmp/secrets.yaml | yq 'select(.kind == "Secret" or .kind == "ExternalSecret") | .kind + "/" + .metadata.name'
